In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# מצב בדיקה מקומי של Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
השתמש במצב בדיקה מקומי (זמין עם `qiskit-ibm-runtime` גרסה v0.22.0 ומעלה) כדי לבדוק תוכניות לפני כיוונון עדין שלהן ושליחתן לחומרה קוונטית אמיתית. לאחר שתאמת את התוכנית שלך במצב בדיקה מקומי, כל מה שעליך לשנות הוא שם ה-Backend כדי להריץ אותה על QPU.

כדי להשתמש במצב בדיקה מקומי, ציין אחד מה-fake backends מ-``qiskit_ibm_runtime.fake_provider`` או ציין Qiskit Aer backend בעת יצירת Qiskit Runtime primitive או session.

- **Fake backends**: ה-[fake backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) ב-``qiskit_ibm_runtime.fake_provider`` מחקים את ההתנהגות של QPUs של IBM&reg; באמצעות תמונות מצב של QPU. תמונות המצב מכילות מידע חשוב על ה-QPU, כגון מפת הצימוד, שערי הבסיס ומאפייני הקיוביטים, שהם שימושיים לבדיקת ה-Transpiler וביצוע סימולציות עם רעש של ה-QPU. מודל הרעש מתמונת המצב מיושם אוטומטית במהלך הסימולציה.
- **סימולטור Aer**: סימולטורים מ-[Qiskit Aer](/guides/simulate-with-qiskit-aer) מספקים סימולציה בעלת ביצועים גבוהים יותר שיכולה לטפל ב-Circuit גדולים יותר ו-[מודלי רעש מותאמים אישית](/guides/build-noise-models). מגוון אפשרויות של שיטות סימולציה זמינות כאשר אתה משתמש ב-`AerSimulator` במצב בדיקה מקומי. ראה את [דוגמת מצב סימולציית Clifford](#clifford-sim), המדגימה כיצד לסמלץ ביעילות Circuit מסוג Clifford עם מספר גדול של קיוביטים.

    <details>
    <summary>**רשימת שיטות הסימולציה הזמינות מ-Qiskit Aer**</summary>

    ראה את תיעוד [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) למידע נוסף.

    * ``"automatic"``: שיטת הסימולציה המוגדרת כברירת מחדל. בחירה אוטומטית של שיטת הסימולציה בהתאם ל-Circuit ומודל הרעש.

    * ``"statevector"``: סימולציה צפופה של וקטור מצב שיכולה לדגום תוצאות מדידה מ-Circuit *אידיאליים* עם כל המדידות בסוף ה-Circuit. עבור סימולציות עם רעש, כל ירייה דוגמת Circuit עם רעש נבחר אקראית ממודל הרעש.

    * ``"density_matrix"``: סימולציה של מטריצת צפיפות שיכולה לדגום תוצאות מדידה מ-Circuit *עם רעש* עם כל המדידות בסוף ה-Circuit.

    * ``"stabilizer"``: סימולטור יעיל של מצב stabilizer מסוג Clifford שיכול לסמלץ Circuit מסוג Clifford עם רעש אם כל השגיאות במודל הרעש הן גם שגיאות Clifford.

    * ``"extended_stabilizer"``: סימולטור קירובי ל-Circuit מסוג Clifford + T המבוסס על פירוק המצב למצב stabilizer מדורג. מספר האיברים גדל עם מספר השערים שאינם Clifford (שערי T).

    * ``"matrix_product_state"``: סימולטור וקטור מצב מבוסס-רשת-טנסורים המשתמש בייצוג Matrix Product State (MPS) עבור המצב. ניתן לבצע זאת עם או בלי קיצוץ ממדי קשר MPS, בהתאם לאפשרויות הסימולטור. ברירת המחדל היא ללא קיצוץ.

    * ``"unitary"``: סימולציה צפופה של מטריצה יוניטרית של Circuit אידיאלי. שיטה זו מסמלצת את המטריצה היוניטרית של ה-Circuit עצמו, ולא את האבולוציה של מצב קוונטי התחלתי. שיטה זו יכולה לסמלץ שערים בלבד; היא אינה תומכת במדידה, איפוס או רעש.

    * ``"superop"``: סימולציה צפופה של מטריצת superoperator של Circuit אידיאלי או עם רעש. שיטה זו מסמלצת את מטריצת ה-superoperator של ה-Circuit עצמו, ולא את האבולוציה של מצב קוונטי התחלתי. שיטה זו יכולה לסמלץ שערים ואיפוסים אידיאליים ועם רעש, אך אינה תומכת במדידות.

    * ``"tensor_network"``: סימולציה מבוססת-רשת-טנסורים התומכת גם בוקטור מצב וגם במטריצת צפיפות. כרגע זמינה רק עבור GPU ומואצת באמצעות ממשקי ה-API של cuQuantum `cuTensorNet`.
    </details>

> **Note:** - תוכל לציין את כל אפשרויות Qiskit Runtime במצב בדיקה מקומי. עם זאת, כל האפשרויות למעט shots מתעלמות כאשר מריצים על סימולטור מקומי.
> - מומלץ להתקין את Qiskit Aer לפני השימוש ב-fake backends או סימולטורי Aer על-ידי הרצת `pip install qiskit-aer`. ה-fake backends משתמשים בסימולטורי Aer מתחת למכסה אם הם זמינים, כדי לנצל את ביצועיהם.


## דוגמת Fake backends

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## דוגמאות AerSimulator
דוגמה עם sessions, ללא רעש:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

לסימולציה עם רעש, ציין QPU (חומרה קוונטית) והגש אותו ל-Aer. Aer בונה מודל רעש המבוסס על נתוני הכיול של אותו QPU, ויוצר Aer backend עם מודל זה. אם תרצה, תוכל [לבנות מודל רעש](/guides/build-noise-models).

> **Caution:** QPU יכול להיות מושפע מסוגים שונים של רעש. מודל הרעש של Qiskit Aer המשמש כאן מסמלץ רק חלק מהם ולכן סביר שיהיה פחות חמור מהרעש ב-QPU אמיתי.
> 
> לפרטים על אילו שגיאות נכללות בעת אתחול מודל רעש מ-QPU, ראה את הפניית ה-API של Aer ל-[`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

דוגמה עם רעש:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### סימולציית Clifford
מכיוון ש-Circuit מסוג Clifford ניתן לסמלץ ביעילות עם תוצאות שניתן לאמת, סימולציית Clifford היא כלי שימושי מאוד. לדוגמה מעמיקה, ראה [סימולציה יעילה של Circuit מסוג stabilizer עם Qiskit Aer primitives](/guides/simulate-stabilizer-circuits).

דוגמה:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## השלבים הבאים
> **Tip:** - עיין בדוגמאות מפורטות של [primitives.](/guides/primitives-examples)
>     - קרא את [מעבר ל-V2 primitives](/guides/v2-primitives).
>     - תרגל עם primitives דרך [שיעור פונקציית העלות](/learning/courses/variational-algorithm-design/cost-functions) ב-IBM Quantum Learning.
>     - למד כיצד לבצע Transpile מקומי בסעיף [Transpile](/guides/transpile).
>     - נסה את מדריך [השוואת הגדרות Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).